<a href="https://colab.research.google.com/github/utkarsh-garg-29/amazon-reviews-bert-sentiment/blob/main/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers datasets torch scikit-learn -q

import pandas as pd
import numpy as np
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/projects/nlp_project_01/Reviews.csv')
print(df.shape)
df.head()

Mounted at /content/drive
(568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [5]:
df = df[df['Score'] != 3]

df['label'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

df = df[['Text', 'label']]

positive = df[df['label'] == 1].sample(n=10000, random_state=42)
negative = df[df['label'] == 0].sample(n=10000, random_state=42)

df_balanced = pd.concat([positive, negative]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape)
print(df_balanced['label'].value_counts())
df_balanced.head()

(20000, 2)
label
0    10000
1    10000
Name: count, dtype: int64


,Text,label
0,I ordered 3 boxes of 18 each.............and o...,0
1,I use red clover tea a lot and so do my friend...,1
2,I was concerned after buying this due to the n...,1
3,If you look forward to kicking back and the en...,1
4,"This should be sold as a medium roast coffee, ...",0


In [6]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['Text'].tolist(),
    df_balanced['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['label']
)

print(f"Train size: {len(train_texts)}")
print(f"Test size: {len(test_texts)}")

Train size: 16000
Test size: 4000


In [7]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

sample = tokenizer(train_texts[0], truncation=True, padding='max_length', max_length=128)
print("Original text:", train_texts[0][:100])
print("\nToken IDs (first 20):", sample['input_ids'][:20])
print("\nDecoded back:", tokenizer.decode(sample['input_ids'][:20]))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original text: I was looking forward to trying this since I've heard good things about Raven's Brew. It was offered

Token IDs (first 20): [101, 1045, 2001, 2559, 2830, 2000, 2667, 2023, 2144, 1045, 1005, 2310, 2657, 2204, 2477, 2055, 10000, 1005, 1055, 24702]

Decoded back: [CLS] i was looking forward to trying this since i ' ve heard good things about raven ' s brew


In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding='max_length', max_length=128)

print("Train encodings keys:", train_encodings.keys())
print("Number of train samples encoded:", len(train_encodings['input_ids']))

In [9]:
import torch

class AmazonReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = AmazonReviewDataset(train_encodings, train_labels)
test_dataset = AmazonReviewDataset(test_encodings, test_labels)

print("Train dataset size:", len(train_dataset))
print("Sample item keys:", train_dataset[0].keys())

Train dataset size: 16000
Sample item keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.to('cuda')

print(model)


In [11]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer ready!")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready!


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.192989,0.284243,0.904500,0.950445,0.853500,0.899368
2,0.114574,0.227130,0.932500,0.936428,0.928000,0.932195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.22920255732536315, metrics={'train_runtime': 378.2087, 'train_samples_per_second': 84.609, 'train_steps_per_second': 5.288, 'total_flos': 1059739189248000.0, 'train_loss': 0.22920255732536315, 'epoch': 2.0})

In [13]:
def predict_sentiment(text):
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt').to('cuda')
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "Positive" if prediction == 1 else "Negative"

test_sentences = [
    "nothing is better than this product",
    "couldn't ask for a better purchase",
    "this is not bad at all",
    "I was not disappointed",
    "wow, what a fantastic waste of money",  # sarcasm
]

for sent in test_sentences:
    print(f"'{sent}' -> {predict_sentiment(sent)}")

'nothing is better than this product' -> Negative
'couldn't ask for a better purchase' -> Positive
'this is not bad at all' -> Positive
'I was not disappointed' -> Positive
'wow, what a fantastic waste of money' -> Negative


In [14]:
model.save_pretrained('./sentiment_bert_model')
tokenizer.save_pretrained('./sentiment_bert_model')


!du -sh ./sentiment_bert_model

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

257M	./sentiment_bert_model
